<p align="center"><a href="https://www.plus2net.com/python/pandas-read_clipboard.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas read_clipboard(): Read Pasted Tables and Desktop Clipboard Text

Run cells in order. Data is embedded and synthetic. Portable text/file examples run in Colab; native clipboard calls are desktop-only and disabled by default. Colab cannot directly access your Windows clipboard. Excel examples need openpyxl. Temporary files are cleaned automatically. Try the exercise before its solution.

## Read pasted table text in Colab

The synthetic Ravi, Raju and Alex table preserves the original marks. Paste tabular text into a string and parse it with read_csv(StringIO(...)). This portable route uses no operating-system clipboard. Use tabs for spreadsheet cells so spaces inside names stay intact. See <a href="pandas-read_csv.php">read_csv()</a>.

In [ ]:
import pandas as pd
import tempfile
import sqlite3
from pathlib import Path
from io import StringIO
pasted = 'NAME\tID\tMATH\tENGLISH\nRavi\t1\t30\t20\nRaju\t2\t40\t30\nAlex\t3\t50\t40\n'
df = pd.read_csv(StringIO(pasted), sep="	")
print(df)
assert df["NAME"].tolist() == ["Ravi", "Raju", "Alex"]

**Expected output**

```text
   NAME  ID  MATH  ENGLISH
0  Ravi   1    30       20
1  Raju   2    40       30
2  Alex   3    50       40
```

## Read the desktop clipboard

On local Python, copy a table from a spreadsheet or the <a href="pandas-dataframe.php">DataFrame example</a>, then set run_desktop=True below. The expected table depends on your copied content. The default separator is a whitespace regex, not a single space. For spreadsheet text, an explicit tab separator is usually clearer. Desktop access is disabled for portable Run All.

In [ ]:
run_desktop = False  # Set True only in local Python after copying a table.
if run_desktop:
    local_table = pd.read_clipboard(sep="	")
    print(local_table)
else:
    print("Desktop clipboard read is disabled; use the pasted-text examples.")

**Expected output**

```text
Desktop clipboard read is disabled; use the pasted-text examples.
```

## Parse whitespace and keep names with spaces

Default read_clipboard() uses one-or-more whitespace characters. This is useful for simple console tables, but may split multiword fields. The first example has single-word names; the second uses tabs to retain John Deo.

In [ ]:
console = 'NAME ID MATH ENGLISH\nRavi 1 30 20\nRaju 2 40 30\nAlex 3 50 40\n'
simple = pd.read_csv(StringIO(console), sep=r"\s+")
print(simple)
names = pd.read_csv(StringIO('id\tname\n1\tJohn Deo\n'), sep="	")
assert names.loc[0, "name"] == "John Deo"
print(names)

**Expected output**

```text
   NAME  ID  MATH  ENGLISH
0  Ravi   1    30       20
1  Raju   2    40       30
2  Alex   3    50       40
   id      name
0   1  John Deo
```

## Choose an em-dash separator and Python engine

Retain the original function-description delimiter example with embedded practice text. Inspect copied text from the <a href="https://www.php.net/manual/en/ref.strings.php">PHP string manual</a> before parsing. The Unicode em dash and regex separators may require engine="python". This sample describes functions for practice rather than fetching the live manual.

In [ ]:
functions_text = 'function—description\nstrlen—Return string length\ntrim—Remove surrounding whitespace\n'
functions = pd.read_csv(StringIO(functions_text), sep="—", engine="python")
print(functions)
assert functions.columns.tolist() == ["function", "description"]

**Expected output**

```text
  function                    description
0   strlen           Return string length
1     trim  Remove surrounding whitespace
```

## Keep identifiers and validate missing values

read_clipboard() accepts read_csv options such as dtype, header and na_values. Choose identifiers as text before parsing; add headers explicitly when the copied region has none.

In [ ]:
records = '001\t75\n002\tpending\n'
typed = pd.read_csv(StringIO(records), sep="	", header=None,
    names=["id", "mark"], dtype={"id": "string"}, na_values=["pending"])
print(typed)
assert typed["id"].tolist() == ["001", "002"]
assert typed["mark"].isna().sum() == 1

**Expected output**

```text
    id  mark
0  001  75.0
1  002   NaN
```

## Save imported data as CSV and Excel

Preserve the original file-export workflow using portable paths. index=False avoids exporting disposable row numbers. openpyxl is needed for the Excel example. See <a href="pandas-to_csv.php">to_csv()</a> and <a href="pandas-to_excel.php">to_excel()</a>.

In [ ]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    csv_file = Path(folder) / "my_clipboard.csv"
    excel_file = Path(folder) / "my_clipboard.xlsx"
    df.to_csv(csv_file, index=False)
    df.to_excel(excel_file, index=False, engine="openpyxl")
    pd.testing.assert_frame_equal(pd.read_csv(csv_file), df)
    pd.testing.assert_frame_equal(pd.read_excel(excel_file, engine="openpyxl"), df)
    print("CSV and Excel round trips passed")

**Expected output**

```text
CSV and Excel round trips passed
```

## Create an HTML table from imported text

Write an escaped HTML table using <a href="pandas-to_html.php">to_html()</a>. escape=True keeps text values from becoming markup. The output below is the actual generated table source, preserving the original HTML teaching example.<figure><img class="img-fluid" src="images/pandas-to_html.png" alt="HTML table generated from imported clipboard data" itemprop="image"></figure>

In [ ]:
table_html = df.to_html(index=False, escape=True)
print(table_html)
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "my_html.html"
    target.write_text(table_html, encoding="utf-8")
    assert target.read_text(encoding="utf-8") == table_html

**Expected output**

```text
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>NAME</th>
      <th>ID</th>
      <th>MATH</th>
      <th>ENGLISH</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Ravi</td>
      <td>1</td>
      <td>30</td>
      <td>20</td>
    </tr>
    <tr>
      <td>Raju</td>
      <td>2</td>
      <td>40</td>
      <td>30</td>
    </tr>
    <tr>
      <td>Alex</td>
      <td>3</td>
      <td>50</td>
      <td>40</td>
    </tr>
  </tbody>
</table>
```

## Common clipboard import pitfalls

<p>Copy a rectangular table with a consistent delimiter. Avoid merged cells, extra titles and footnotes. sep and engine are parsing choices; changing engine does not repair a missing clipboard service. A local/headless backend may report PyperclipException when unavailable. Colab runs remotely and cannot read your Windows clipboard; paste text or upload a CSV/Excel file instead. Clipboard data is transient, so save useful imports. Verify row counts, headers, dtypes and missing values before analysis. A headerless table needs header=None and names. Regex separators can interact poorly with quoted delimiters; prefer a standard delimiter where possible.</p>

## Exercise: parse functions and export them

Parse the em-dash function text above, then write an Excel file without an index. Verify the two function names. This retains the original PHP-function worksheet exercise.

## Exercise solution

Use the Python parser for the supplied em-dash text, export and read the worksheet back.

In [ ]:
answer = pd.read_csv(StringIO(functions_text), sep="—", engine="python")
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    target = Path(folder) / "php_functions.xlsx"
    answer.to_excel(target, index=False, engine="openpyxl")
    checked = pd.read_excel(target, engine="openpyxl")
    assert checked["function"].tolist() == ["strlen", "trim"]
    print(checked)

**Expected output**

```text
  function                    description
0   strlen           Return string length
1     trim  Remove surrounding whitespace
```